# 01 — Data Extraction and Dataset Split

**Project:** AuraVision — Crowd Age Group Majority Classifier  
**Purpose:** collect images, split the dataset into `dataset/train/` and `dataset/test/`, and save everything to Google Drive.

All data is saved directly to the shared Google Drive folder so nothing is lost when the Colab session ends.

## Colab setup

Run this first in Google Colab. If you uploaded the ZIP, extract it and open the notebook from the extracted project folder.

In [ ]:
!pip install -q torch torchvision pillow numpy matplotlib seaborn scikit-learn requests

import os
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))

## Step 1 — Mount Google Drive

All downloaded images and split folders will be saved directly to the shared Drive folder so they persist after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Shared project folder on Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/AuraVision"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root ready: {DRIVE_ROOT}")

## Step 2 — Download all images into `crowd_dataset/`

Both the training and test query sets are combined into a single extraction run. Test-specific queries are listed first inside each class so they are easy to identify. All images land in `crowd_dataset/<Class>/` inside the Drive folder.

**Security note:** The Pexels API key below is visible in plain text. Regenerate it before sharing this notebook publicly.

In [ ]:
import requests
import os

# ── Configuration
API_KEY  = '45fFNqCjN4SWAHfgFxsRGTqrx1sZVoml5EFSpOdX0y9Utxyf96cujulx'
SAVE_DIR = os.path.join(DRIVE_ROOT, 'extracted_data')

# Test-specific queries are placed at the top of each class list.
# Training queries follow beneath them — all images go into one folder.
CLASSES = {
    "Children": [
        # test-specific queries
        "Elementary school assembly",
        "Family day at zoo",
        "Kids birthday party background",
        "Youth soccer match crowd",
        # training queries
        "children",
        "children at school",
        "playground children",
        "children playing",
        "kindergarten",
        "youth sports",
    ],
    "Adults": [
        # test-specific queries
        "City commuters morning",
        "Airport terminal crowd",
        "Tech conference audience",
        "Nightlife street scene",
        "Busy food court",
        # training queries
        "business women",
        "business men",
        "university students",
        "new married couples",
        "adults working",
        "music festival adults",
        "adults in gym",
        "airport terminal",
    ],
    "Seniors": [
        # test-specific queries
        "Public park morning walkers",
        "Senior center ballroom",
        "Older people gardening together",
        "Pensioners traveling in tour group",
        # training queries
        "seniors",
        "seniors walking",
        "retirement community",
        "gardening seniors",
        "seniors socializing",
    ],
}
IMAGES_PER_QUERY = 40  # adjust based on your needs
# ──────────────────────────────────────────────────────────────────────────────


def download_images():
    headers = {'Authorization': API_KEY}

    for label, queries in CLASSES.items():
        class_path = os.path.join(SAVE_DIR, label)
        os.makedirs(class_path, exist_ok=True)

        for query in queries:
            url = f"https://api.pexels.com/v1/search?query={query}&per_page={IMAGES_PER_QUERY}"
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                data = response.json()
                for i, photo in enumerate(data['photos']):
                    img_url  = photo['src']['large']
                    img_data = requests.get(img_url).content
                    filename = f"{query.replace(' ', '_')}_{i}.jpg"
                    with open(os.path.join(class_path, filename), 'wb') as f:
                        f.write(img_data)
                print(f"  ✓ '{query}' → {len(data['photos'])} images saved to '{label}/'")
            else:
                print(f"  ✗ Error {response.status_code} for query '{query}'")


print(f"Saving images to: {SAVE_DIR}\n")
download_images()
print("\nDownload complete.")

# Summary
for label in CLASSES:
    class_path = os.path.join(SAVE_DIR, label)
    count = len([f for f in os.listdir(class_path) if f.endswith('.jpg')])
    print(f"  {label}: {count} images")

## Step 3 — Split into `dataset/train/` and `dataset/test/`

Images are split 80 % train / 20 % test directly into `dataset/` inside the Drive folder. Both subfolders keep the class subfolder structure so they work with PyTorch's `ImageFolder` out of the box. No intermediate folders are created.

In [ ]:
import shutil
import random

SOURCE_DIR = os.path.join(DRIVE_ROOT, 'extracted_data')   # all downloaded images
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'dataset')         # final split destination
SPLIT_RATIO = 0.8                                        # 80 % train, 20 % test
RANDOM_SEED = 42

random.seed(RANDOM_SEED)


def split_dataset(source_dir, output_dir, split_ratio):
    classes = [d for d in os.listdir(source_dir)
               if os.path.isdir(os.path.join(source_dir, d))]

    # Create destination subfolders up front
    for subset in ('train', 'test'):
        for class_name in classes:
            os.makedirs(os.path.join(output_dir, subset, class_name), exist_ok=True)

    total_train = total_test = 0

    for class_name in sorted(classes):
        class_path = os.path.join(source_dir, class_name)
        images = sorted([
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f))
            and f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))
        ])
        random.shuffle(images)

        split_point  = int(len(images) * split_ratio)
        train_images = images[:split_point]
        test_images  = images[split_point:]

        for fname in train_images:
            shutil.copy(
                os.path.join(class_path, fname),
                os.path.join(output_dir, 'train', class_name, fname)
            )
        for fname in test_images:
            shutil.copy(
                os.path.join(class_path, fname),
                os.path.join(output_dir, 'test', class_name, fname)
            )

        print(f"  {class_name}: {len(train_images)} train  |  {len(test_images)} test")
        total_train += len(train_images)
        total_test  += len(test_images)

    print(f"\n  Total → train: {total_train}  |  test: {total_test}")


print(f"Splitting '{SOURCE_DIR}' → '{OUTPUT_DIR}'\n")
split_dataset(SOURCE_DIR, OUTPUT_DIR, SPLIT_RATIO)
print(f"\nSplit complete. Files saved to Drive under: {OUTPUT_DIR}")

## Output of this notebook

After running this notebook, your Google Drive folder will contain:

```text
AuraVision/
├── extracted_data/          # all downloaded images (kept for reference)
│   ├── Children/
│   ├── Adults/
│   └── Seniors/
└── dataset/                # ready-to-use split for training and evaluation
    ├── train/
    │   ├── Children/
    │   ├── Adults/
    │   └── Seniors/
    └── test/
        ├── Children/
        ├── Adults/
        └── Seniors/
```

Both `dataset/train/` and `dataset/test/` use the `ImageFolder` structure expected by notebooks 02, 03, and 04.